# Serving an ML Model with Flask

Flask is a lightweight synchronous web framework — battle-tested, simple, and widely deployed. In this notebook you will train a classifier, wrap it in a Flask API, and test it entirely inside the notebook using Flask's built-in test client (no live server needed).

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain when to choose Flask vs FastAPI for model serving
2. Build a Flask app with `/predict` and `/health` endpoints
3. Add input validation and return correct HTTP status codes (400 / 500)
4. Test a Flask app in-notebook using `app.test_client()`
5. Describe how to run Flask with Gunicorn in production

## 1. Flask vs FastAPI — Quick Comparison

| | Flask | FastAPI |
|---|---|---|
| Style | Synchronous (WSGI) | Async-first (ASGI) |
| Docs | Manual | Auto OpenAPI / Swagger |
| Validation | Manual | Pydantic (automatic) |
| Maturity | ~15 years, massive ecosystem | ~5 years, fast-growing |
| Best for | Simple APIs, legacy codebases | High-throughput, data validation |

Choose Flask when your team already uses it or when you need a simpler mental model. FastAPI is covered in the next notebook.

## 2. Train and Save the Model

We train a RandomForest on the Iris dataset and save it with `joblib`. This happens once; the API loads the saved artifact at startup.

In [ ]:
import joblib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

acc = accuracy_score(y_test, clf.predict(X_test))
print(f"Test accuracy: {acc:.2%}")

MODEL_PATH = "/tmp/iris_rf.joblib"
joblib.dump(clf, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

## 3. Write the Flask Application

`%%writefile` saves the cell content to `app.py`. The key design decisions:
- Load the model **once** at startup (not inside the predict function)
- Return **400** for bad input (client's fault), **500** for unexpected errors (server's fault)
- Keep the route handlers thin — delegate logic to helper functions

In [ ]:
%%writefile /tmp/flask_app.py
import joblib
import numpy as np
from flask import Flask, request, jsonify

app = Flask(__name__)

# --- Load model once at startup ---
MODEL_PATH = "/tmp/iris_rf.joblib"
model = joblib.load(MODEL_PATH)
CLASSES = ["setosa", "versicolor", "virginica"]
REQUIRED_FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

def validate_input(data):
    """Return (features_array, error_message). error_message is None if valid."""
    if not isinstance(data, dict):
        return None, "Request body must be a JSON object"
    missing = [f for f in REQUIRED_FEATURES if f not in data]
    if missing:
        return None, f"Missing fields: {missing}"
    try:
        features = np.array([[float(data[f]) for f in REQUIRED_FEATURES]])
    except (TypeError, ValueError) as e:
        return None, f"All fields must be numeric: {e}"
    return features, None

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": MODEL_PATH}), 200

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True)
    features, error = validate_input(data)
    if error:
        return jsonify({"error": error}), 400          # 400 = client's fault
    try:
        prediction = model.predict(features)[0]
        probabilities = model.predict_proba(features)[0]
        return jsonify({
            "prediction": CLASSES[prediction],
            "class_id": int(prediction),
            "confidence": round(float(probabilities.max()), 3)
        }), 200
    except Exception as e:
        return jsonify({"error": "Internal server error", "detail": str(e)}), 500  # 500 = server's fault

if __name__ == "__main__":
    app.run(debug=True, port=5000)

## 4. Test with Flask's Test Client

`app.test_client()` lets you send HTTP requests to your Flask app without starting a real server. This is how you write tests and also how we experiment in-notebook.

In [ ]:
import sys
import importlib
import json

# Import the app we just wrote
sys.path.insert(0, "/tmp")
import flask_app as flask_module
importlib.reload(flask_module)       # reload so joblib path is fresh
app = flask_module.app
client = app.test_client()

# --- Test 1: Health check ---
resp = client.get("/health")
print("Health check:", resp.status_code, resp.get_json())

# --- Test 2: Valid prediction ---
payload = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
resp = client.post("/predict", json=payload)
print("Valid request:", resp.status_code, resp.get_json())

# --- Test 3: Missing field (expect 400) ---
bad_payload = {"sepal_length": 5.1, "sepal_width": 3.5}  # missing two fields
resp = client.post("/predict", json=bad_payload)
print("Missing fields:", resp.status_code, resp.get_json())

# --- Test 4: Non-numeric value (expect 400) ---
bad_payload2 = {"sepal_length": "big", "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
resp = client.post("/predict", json=bad_payload2)
print("Non-numeric:  ", resp.status_code, resp.get_json())

## 5. Batch Testing — Run Multiple Requests

Below we simulate calling the API for every sample in the test set and compare the API's responses against sklearn's direct predictions.

In [ ]:
CLASSES = ["setosa", "versicolor", "virginica"]

api_predictions = []
for row in X_test:
    payload = {
        "sepal_length": row[0], "sepal_width": row[1],
        "petal_length": row[2], "petal_width": row[3]
    }
    resp = client.post("/predict", json=payload)
    api_predictions.append(resp.get_json()["class_id"])

direct_predictions = clf.predict(X_test).tolist()

match = sum(a == d for a, d in zip(api_predictions, direct_predictions))
print(f"API predictions match sklearn direct: {match}/{len(X_test)}")
print("Sample API responses:")
for i in range(3):
    print(f"  Row {i}: api={CLASSES[api_predictions[i]]}, direct={CLASSES[direct_predictions[i]]}")

## 6. Production Deployment with Gunicorn

Flask's built-in development server is single-threaded and not safe for production. Gunicorn is a production-grade WSGI server that spawns multiple worker processes.

```bash
# Install
pip install gunicorn

# Run with 4 worker processes (rule of thumb: 2 * CPU cores + 1)
gunicorn flask_app:app --workers 4 --bind 0.0.0.0:5000

# Test with curl
curl -X POST http://localhost:5000/predict \
     -H "Content-Type: application/json" \
     -d '{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'
```

**Gunicorn vs dev server:**
- Dev server: 1 thread, restarts on code change, never use in production
- Gunicorn: multiple workers, stable, handles concurrent requests

## 7. Summary

In this notebook you:
- Trained a RandomForest and saved it with `joblib`
- Built a Flask app that loads the model once at startup and serves predictions via `/predict`
- Added input validation returning **400** for bad client input and **500** for server errors
- Tested the entire API in-notebook using `app.test_client()` — no live server needed
- Learned how to run Flask with Gunicorn for production traffic

The Flask serving pattern is: **train → save artifact → load once at startup → validate input → predict → return JSON**.

## Self-Check (answer before scrolling up)

1. **What does `app.test_client()` give you?** (Hint: think about what you avoided by using it)
2. **What is the difference between Flask's dev server and Gunicorn?** Name at least two differences.
3. **Why return 400 for bad input instead of 500?** What does each status code communicate to the caller?